In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, "..")                       # so Python can find data_prep.py
from data_prep import load_audio

ROOT = Path("..").resolve()
train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")      # test.csv stays closed until the very end

label2id = json.load(open("../data/processed/label2id.json"))
id2label = {v: k for k, v in label2id.items()}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| train:", len(train_df), "| val:", len(val_df), "| classes:", len(label2id))

device: cpu | train: 1004 | val: 215 | classes: 28


In [2]:
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2ForSequenceClassification

MODEL = "facebook/wav2vec2-base"     # small model for your laptop; on Colab use "facebook/wav2vec2-xls-r-300m"

# The feature extractor prepares raw audio for the model (normalization)
fe = Wav2Vec2FeatureExtractor.from_pretrained(MODEL)

# The model: pretrained Wav2Vec2 + a NEW classification layer with 28 outputs
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=28,
    label2id=label2id,
    id2label=id2label,
).to(device).eval()

# Try it on one training file (the model is NOT trained yet)
r = train_df.iloc[0]
audio, sr = load_audio(str(ROOT / r["file_path"]))
x = fe(audio, sampling_rate=16000, return_tensors="pt").input_values.to(device)
print("input shape:", x.shape, "| mean:", round(x.mean().item(), 3), "| std:", round(x.std().item(), 3))

with torch.no_grad():
    logits = model(x).logits
print("logits shape:", logits.shape)
print("true:", r["label"], "| predicted:", id2label[int(logits.argmax())])

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

c:\Users\ALMonir\OneDrive\Desktop\college\braile\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ALMonir\.cache\huggingface\hub\models--facebook--wav2vec2-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

c:\Users\ALMonir\OneDrive\Desktop\college\braile\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ALMonir\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  380MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  380MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
projector.weight             | MISSING    | 
projector.bias               | MISSING    | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


input shape: torch.Size([1, 16000]) | mean: -0.0 | std: 1.0
logits shape: torch.Size([1, 28])
true: Noon | predicted: Sheen


# Conclusion (cell 2): loading Wav2Vec2 before training
# - The pretrained facebook/wav2vec2-base model (~380 MB) was downloaded once from the
#   Hugging Face Hub and cached locally; later runs will load it in seconds.
# - The feature extractor normalized one 1-second recording to shape (1, 16000)
#   with mean ~0 and std ~1, as the model expects.
# - A new classification head (projector + classifier) with 28 outputs, one per Arabic
#   letter, was added on top of the pretrained model. Its weights are randomly
#   initialized and reported as MISSING in the load report. This is expected.
# - The UNEXPECTED keys (quantizer, project_q, project_hid) belong to the original
#   self-supervised pre-training and are not needed for classification, so they were ignored.
# - Before training, the prediction is random (Noon predicted as Sheen), which is expected
#   because the classification head has not learned anything yet.
# - Output shape (1, 28) confirms the model returns one score per class.
# Next step: build the Dataset/DataLoader, then fine-tune and compare against the
# MFCC + SVM baseline (val accuracy 0.698, macro-F1 0.725).

In [ ]:
#getting the data ready for the training and validation sets

from torch.utils.data import Dataset, DataLoader

class LetterDataset(Dataset):
    """Reads one audio file and returns (normalized audio, label number)."""

    def __init__(self, df, augment=False):
        self.df = df.reset_index(drop=True)      # make row numbers 0..n-1
        self.augment = augment                   # True only for the training set

    def __len__(self):
        return len(self.df)                      # how many files

    def __getitem__(self, i):
        row = self.df.iloc[i]
        audio, _ = load_audio(str(ROOT / row["file_path"]))
        if self.augment:
            audio = np.roll(audio, np.random.randint(-1600, 1600))                         # shift in time (up to 0.1 s)
            audio = audio + np.random.randn(len(audio)).astype(np.float32) * 0.003 * np.random.rand()   # light noise
        x = fe(audio, sampling_rate=16000, return_tensors="pt").input_values[0]
        return x, int(row["label_id"])

def collate(batch):
    xs, ys = zip(*batch)                         # split the list of pairs into audios and labels
    return torch.stack(xs), torch.tensor(ys)     # stack into one batch tensor

train_ds = LetterDataset(train_df, augment=True)
val_ds = LetterDataset(val_df, augment=False)
train_dl = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate)
val_dl = DataLoader(val_ds, batch_size=8, shuffle=False, collate_fn=collate)

xb, yb = next(iter(train_dl))                    # take one batch to check shapes
print(xb.shape, yb.shape)                        # expect: torch.Size([8, 16000]) torch.Size([8])
print([id2label[int(i)] for i in yb])

torch.Size([8, 16000]) torch.Size([8])
['Ba', 'Ain', 'Ha2', 'Ya', 'Zaa', 'Ha2', 'Thal', 'Zay']


In [ ]:
# Evaluate the model before training to measure its initial performance.
# This gives us a baseline to compare against after training.
# We use the validation set to check how well the model predicts unseen audio files.
# torch.no_grad() disables gradient computation because we are only evaluating,
# not updating the model's weights.
# After training, we can compare accuracy and macro-F1 to see if performance improved.

In [4]:
from sklearn.metrics import accuracy_score, f1_score

@torch.no_grad() # disable gradient computation (faster, less memory)
def evaluate(model, loader):
    """Run the model on a whole DataLoader and return (true labels, predicted labels)."""
    model.eval()                                   # evaluation mode (dropout off)
    preds, trues = [], []
    for x, y in loader:                            # loop over batches
        logits = model(x.to(device)).logits        # (batch, 28) scores
        preds += logits.argmax(dim=-1).cpu().tolist()   # highest score = predicted letter id
        trues += y.tolist()
    return np.array(trues), np.array(preds)

# Untrained model on the validation set: should be near chance level (1/28 = 0.036)
y_true, y_pred = evaluate(model, val_dl)
print("before training | val acc:", round(accuracy_score(y_true, y_pred), 3),
      "| macro-F1:", round(f1_score(y_true, y_pred, average="macro"), 3))

before training | val acc: 0.028 | macro-F1: 0.01


In [5]:
def train(lr=3e-5, epochs=15, augment=True, limit=None, save_dir=None, seed=42):
    """Fine-tune a fresh Wav2Vec2 classifier. Returns (best val macro-F1, history)."""
    torch.manual_seed(seed); np.random.seed(seed)             # same random start every run

    df = train_df.sample(limit, random_state=0) if limit else train_df   # optional small subset for testing
    tr_dl = DataLoader(LetterDataset(df, augment=augment), batch_size=8, shuffle=True, collate_fn=collate)

    m = Wav2Vec2ForSequenceClassification.from_pretrained(
        MODEL, num_labels=28, label2id=label2id, id2label=id2label).to(device)
    m.freeze_feature_encoder()                                # keep the low-level audio CNN fixed

    params = [p for p in m.parameters() if p.requires_grad]   # only the trainable weights
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)

    best_f1, history = -1.0, []
    for epoch in range(1, epochs + 1):
        m.train()                                             # training mode (dropout on)
        total_loss = 0.0
        for x, y in tr_dl:
            loss = m(x.to(device), labels=y.to(device)).loss  # forward pass + loss
            opt.zero_grad()                                   # clear old gradients
            loss.backward()                                   # compute new gradients
            opt.step()                                        # update the weights
            total_loss += loss.item()

        y_true, y_pred = evaluate(m, val_dl)                  # check on validation after each epoch
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average="macro")
        history.append((epoch, total_loss / len(tr_dl), acc, f1))
        print(f"epoch {epoch:2d} | loss {total_loss/len(tr_dl):.3f} | val acc {acc:.3f} | val macro-F1 {f1:.3f}")

        if f1 > best_f1:                                      # keep only the best version
            best_f1 = f1
            if save_dir:
                m.save_pretrained(save_dir); fe.save_pretrained(save_dir)
    return best_f1, history

In [6]:
best, hist = train(limit=48, epochs=1)      # 6 batches, 1 epoch: only checks that the code runs
print("best macro-F1:", best)

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
projector.weight             | MISSING    | 
projector.bias               | MISSING    | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


: 